# Classical Machine-Learning Experiments

This notebook compares SVM, Random Forest, and Logistic Regression classifiers trained on frozen intermediate features from the **already-trained accident-detection MobileNetV2 and ResNet50 models** in `models/`. It does not initialize ImageNet-only replacement models or modify the CNNs.

The CNN notebooks use 224×224 RGB inputs, their architecture-specific `preprocess_input` functions, and `class_names=["Accident", "Non Accident"]`. This notebook uses the same preprocessing and maps **Accident = 0 (positive class)** and **Non Accident = 1**. The saved CNN metrics use the CNN notebooks' default binary positive label (class 1), so those metrics are labeled separately in the comparison output.

Run the cells from top to bottom. Feature extraction is cached under `results/classical_ml/features/`; classifiers and evaluation outputs are saved under `results/classical_ml/`.

In [ ]:
import hashlib
import importlib.util
import json
import random
import sys
import time
import zipfile
import tempfile
from pathlib import Path

PACKAGE_MODULES = {
    "tensorflow": "tensorflow", "numpy": "numpy", "pandas": "pandas",
    "matplotlib": "matplotlib", "scikit-learn": "sklearn", "joblib": "joblib",
    "pillow": "PIL", "jupyter": "jupyter",
}
missing_packages = [name for name, module in PACKAGE_MODULES.items()
                    if importlib.util.find_spec(module) is None]
if missing_packages:
    install = f"{sys.executable} -m pip install tensorflow numpy pandas matplotlib scikit-learn joblib pillow jupyter"
    raise ModuleNotFoundError(f"Missing packages: {missing_packages}. Install with: {install}")

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score, roc_auc_score)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

search_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
PROJECT_ROOT = next((root for root in search_roots
                     if (root / "datasets" / "data").is_dir() and (root / "models").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root. Run from the project or notebooks directory.")

DATA_DIR = PROJECT_ROOT / "datasets" / "data"
MODEL_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results" / "classical_ml"
FEATURES_DIR = RESULTS_DIR / "features"
CLASSIFIERS_DIR = RESULTS_DIR / "models"
CONFUSION_DIR = RESULTS_DIR / "confusion_matrices"
REPORTS_DIR = RESULTS_DIR / "reports"
PREDICTIONS_DIR = RESULTS_DIR / "test_predictions"
for folder in (FEATURES_DIR, CLASSIFIERS_DIR, CONFUSION_DIR, REPORTS_DIR, PREDICTIONS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

IMG_SIZE = (224, 224)
CLASS_NAMES = ["Accident", "Non Accident"]
CLASS_TO_LABEL = {"Accident": 0, "Non Accident": 1}
LABEL_TO_CLASS = {label: name for name, label in CLASS_TO_LABEL.items()}
POSITIVE_LABEL = CLASS_TO_LABEL["Accident"]
SPLITS = ("train", "val", "test")
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}
MODEL_PATHS = {
    "MobileNetV2": MODEL_DIR / "mobilenetv2_best.keras",
    "ResNet50": MODEL_DIR / "resnet50_best.keras",
}

for split in SPLITS:
    split_dir = DATA_DIR / split
    if not split_dir.is_dir():
        raise FileNotFoundError(f"Dataset split directory not found: {split_dir}")
    for class_name in CLASS_NAMES:
        if not (split_dir / class_name).is_dir():
            raise FileNotFoundError(f"Required class directory not found: {split_dir / class_name}")
for path in MODEL_PATHS.values():
    if not path.is_file():
        raise FileNotFoundError(f"Required trained CNN model not found: {path}")

FILES_BY_SPLIT, LABELS_BY_SPLIT = {}, {}
for split in SPLITS:
    files, labels = [], []
    for class_name, label in CLASS_TO_LABEL.items():
        class_files = [path for path in (DATA_DIR / split / class_name).iterdir()
                       if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS]
        class_files.sort(key=lambda path: path.as_posix().casefold())
        files.extend(class_files)
        labels.extend([label] * len(class_files))
    order = sorted(range(len(files)), key=lambda index: files[index].as_posix().casefold())
    FILES_BY_SPLIT[split] = [files[index] for index in order]
    LABELS_BY_SPLIT[split] = np.asarray([labels[index] for index in order], dtype=np.int64)
    if not files or set(np.unique(LABELS_BY_SPLIT[split])) != {0, 1}:
        raise ValueError(f"Both classes must have supported images in the {split!r} split.")
    counts = np.bincount(LABELS_BY_SPLIT[split], minlength=2)
    print(f"{split}: Accident={counts[0]}, Non Accident={counts[1]}, total={len(files)}")

print("Project root:", PROJECT_ROOT)
print("Saved accident-trained models:", MODEL_PATHS)
print("Class mapping:", CLASS_TO_LABEL, "| input: 224 x 224 RGB")

## Feature extraction from saved accident-trained CNNs

Both archived CNNs were trained on 224×224 RGB images with class order `Accident`, `Non Accident` and a one-unit sigmoid output. Their feature extractors use the saved model's `dropout` layer output—the global-average-pooled feature vector immediately before the final two-class decision—not the sigmoid prediction. Each CNN receives its own original Keras `preprocess_input` function.

The ResNet50 archive contains disabled/empty Keras configuration fields unsupported by some Keras builds. If direct loading hits that specific compatibility error, this cell retries from a temporary copy with only inert configuration metadata removed; the original archive and learned weights are never modified.

In [ ]:
def _sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _adapt_inert_keras_metadata(value, changes):
    if isinstance(value, dict):
        class_name, config = value.get("class_name"), value.get("config")
        if class_name == "GlorotUniform" and isinstance(config, dict):
            for key in ("input_axes", "output_axes"):
                if key in config:
                    if config[key] is not None:
                        raise ValueError(f"Cannot safely adapt non-empty initializer field {key!r}.")
                    del config[key]
                    changes["initializer_fields"] += 1
        if class_name == "BatchNormalization" and isinstance(config, dict):
            if config.get("renorm") is True:
                raise ValueError("Cannot safely adapt BatchNormalization with renorm enabled.")
            if config.get("renorm") is False:
                for key in ("renorm", "renorm_clipping", "renorm_momentum"):
                    if key in config:
                        del config[key]
                        changes["disabled_renorm_fields"] += 1
        if isinstance(config, dict) and config.get("quantization_config", "missing") is None:
            del config["quantization_config"]
            changes["empty_quantization_fields"] += 1
        for child in value.values():
            _adapt_inert_keras_metadata(child, changes)
    elif isinstance(value, list):
        for child in value:
            _adapt_inert_keras_metadata(child, changes)


def _load_saved_cnn(path):
    try:
        return tf.keras.models.load_model(path, compile=False)
    except TypeError as error:
        if "input_axes" not in str(error):
            raise
        changes = {"initializer_fields": 0, "disabled_renorm_fields": 0, "empty_quantization_fields": 0}
        with tempfile.TemporaryDirectory(prefix="accident-cnn-compat-") as temp_dir:
            compatible_path = Path(temp_dir) / path.name
            with zipfile.ZipFile(path, "r") as source:
                config = json.loads(source.read("config.json"))
                _adapt_inert_keras_metadata(config, changes)
                if not any(changes.values()):
                    raise RuntimeError(f"No known inert Keras metadata found in {path.name}.") from error
                with zipfile.ZipFile(compatible_path, "w", zipfile.ZIP_DEFLATED) as target:
                    for entry in source.infolist():
                        content = (json.dumps(config, separators=(",", ":")).encode("utf-8")
                                   if entry.filename == "config.json" else source.read(entry.filename))
                        target.writestr(entry, content)
            model = tf.keras.models.load_model(compatible_path, compile=False)
        print(f"Loaded {path.name} from a temporary compatibility copy; original model unchanged. {changes}")
        return model


PREPROCESSORS = {
    "MobileNetV2": tf.keras.applications.mobilenet_v2.preprocess_input,
    "ResNet50": tf.keras.applications.resnet50.preprocess_input,
}
CNN_MODELS, FEATURE_EXTRACTORS, FEATURE_LAYER_NAMES = {}, {}, {}
for backbone, model_path in MODEL_PATHS.items():
    cnn = _load_saved_cnn(model_path)
    if tuple(cnn.input_shape[1:]) != (224, 224, 3) or cnn.output_shape[-1] != 1:
        raise ValueError(f"Unexpected saved {backbone} architecture: {cnn.input_shape} -> {cnn.output_shape}")
    feature_layer = cnn.get_layer("dropout")
    extractor = tf.keras.Model(cnn.input, feature_layer.output, name=f"{backbone.lower()}_saved_features")
    if len(extractor.output_shape) != 2:
        raise ValueError(f"Expected pooled feature vector from {backbone}; got {extractor.output_shape}.")
    CNN_MODELS[backbone] = cnn
    FEATURE_EXTRACTORS[backbone] = extractor
    FEATURE_LAYER_NAMES[backbone] = feature_layer.name
    print(f"{backbone}: loaded {cnn.name}; feature layer={feature_layer.name}; shape={extractor.output_shape[1:]}")

model_hashes = {name: _sha256_file(path) for name, path in MODEL_PATHS.items()}
dataset_signature = {
    split: [{"path": p.relative_to(PROJECT_ROOT).as_posix(), "size": p.stat().st_size,
             "mtime_ns": p.stat().st_mtime_ns} for p in FILES_BY_SPLIT[split]]
    for split in SPLITS
}
CACHE_METADATA = {
    "schema_version": 1, "class_mapping": CLASS_TO_LABEL, "image_size": list(IMG_SIZE),
    "preprocessing": {"MobileNetV2": "mobilenet_v2.preprocess_input",
                      "ResNet50": "resnet50.preprocess_input"},
    "feature_layers": FEATURE_LAYER_NAMES, "model_sha256": model_hashes, "dataset": dataset_signature,
}
FEATURE_CACHE = FEATURES_DIR / "cnn_feature_cache.npz"
FEATURES, LABELS, cache_is_valid = {}, {}, False
if FEATURE_CACHE.is_file():
    with np.load(FEATURE_CACHE, allow_pickle=False) as archive:
        if json.loads(str(archive["metadata"].item())) == CACHE_METADATA:
            LABELS = {split: archive[f"{split}_labels"].astype(np.int64, copy=True) for split in SPLITS}
            for split in SPLITS:
                for backbone in MODEL_PATHS:
                    key = f"{split}_{backbone.lower()}_features"
                    FEATURES[(backbone, split)] = archive[key].astype(np.float32, copy=True)
            cache_is_valid = True
            print("Loaded validated feature cache:", FEATURE_CACHE)
        else:
            print("Feature cache is stale; recomputing features.")

if not cache_is_valid:
    LABELS = {split: labels.copy() for split, labels in LABELS_BY_SPLIT.items()}
    for backbone, extractor in FEATURE_EXTRACTORS.items():
        for split in SPLITS:
            paths, batches = FILES_BY_SPLIT[split], []
            for start in range(0, len(paths), 16):
                batch_paths = paths[start:start + 16]
                try:
                    images = np.stack([
                        tf.keras.utils.img_to_array(tf.keras.utils.load_img(
                            path, target_size=IMG_SIZE, color_mode="rgb"))
                        for path in batch_paths
                    ]).astype(np.float32)
                except (OSError, ValueError) as error:
                    raise ValueError(f"Could not decode dataset image {batch_paths[0]}: {error}") from error
                inputs = PREPROCESSORS[backbone](images)
                batches.append(np.asarray(extractor.predict(inputs, verbose=0), dtype=np.float32))
            features = np.concatenate(batches, axis=0)
            if features.ndim != 2 or features.shape[0] != len(paths) or not np.isfinite(features).all():
                raise ValueError(f"Invalid {backbone} {split} features: {features.shape}")
            FEATURES[(backbone, split)] = features
            print(f"Extracted {backbone} {split}: {features.shape}")
    cached = {"metadata": np.asarray(json.dumps(CACHE_METADATA, sort_keys=True))}
    for split in SPLITS:
        cached[f"{split}_labels"] = LABELS[split]
        for backbone in MODEL_PATHS:
            cached[f"{split}_{backbone.lower()}_features"] = FEATURES[(backbone, split)]
    np.savez_compressed(FEATURE_CACHE, **cached)
    print("Saved feature cache:", FEATURE_CACHE)

for split in SPLITS:
    if not np.array_equal(LABELS[split], LABELS_BY_SPLIT[split]):
        raise ValueError(f"Feature-cache labels do not align with deterministic {split} file order.")
    for backbone in MODEL_PATHS:
        values = FEATURES[(backbone, split)]
        if values.shape[0] != len(FILES_BY_SPLIT[split]) or not np.isfinite(values).all():
            raise ValueError(f"Invalid cached {backbone} {split} features: {values.shape}")
        counts = np.bincount(LABELS[split], minlength=2).tolist()
        print(f"Verified {backbone} {split}: shape={values.shape}; labels [Accident, Non Accident]={counts}")

## Fit six classifiers; select using validation data

Classifiers are fit only on training features. StandardScaler is inside the SVM and Logistic Regression pipelines, so it is fit on training data only. The six fixed configurations are ranked by validation Accident-class F1, followed by accuracy and ROC-AUC; no hyperparameter search is performed. After selection, each experiment is evaluated on the held-out test split.

ROC-AUC scores are oriented so Accident is the positive class even though its dataset label is 0. SVM uses a decision score; the other classifiers use the Accident probability. Test prediction times cover `predict` calls.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay


def _accident_scores(classifier, features):
    classes = np.asarray(classifier.classes_)
    if len(classes) != 2 or POSITIVE_LABEL not in classes:
        raise ValueError(f"Expected binary classes including Accident label 0; got {classes}.")
    accident_index = int(np.flatnonzero(classes == POSITIVE_LABEL)[0])
    if hasattr(classifier, "predict_proba"):
        return classifier.predict_proba(features)[:, accident_index], "probability"
    if hasattr(classifier, "decision_function"):
        score = np.asarray(classifier.decision_function(features)).reshape(-1)
        return (score if classes[1] == POSITIVE_LABEL else -score), "decision_score"
    return None, None


def _metrics(y_true, y_pred, accident_score):
    result = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, pos_label=POSITIVE_LABEL, zero_division=0),
        "recall": recall_score(y_true, y_pred, pos_label=POSITIVE_LABEL, zero_division=0),
        "f1": f1_score(y_true, y_pred, pos_label=POSITIVE_LABEL, zero_division=0),
        "roc_auc": None,
    }
    binary_accident = (np.asarray(y_true) == POSITIVE_LABEL).astype(np.int8)
    if accident_score is not None and np.unique(binary_accident).size == 2:
        result["roc_auc"] = roc_auc_score(binary_accident, accident_score)
    return result


CLASSIFIERS = {
    "SVM": lambda: make_pipeline(
        StandardScaler(), SVC(kernel="rbf", C=1.0, class_weight="balanced",
                              probability=False, random_state=SEED)),
    "RandomForest": lambda: RandomForestClassifier(
        n_estimators=300, class_weight="balanced_subsample", random_state=SEED, n_jobs=-1),
    "LogisticRegression": lambda: make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=3000, class_weight="balanced", random_state=SEED)),
}
fitted, validation_rows = {}, []
for backbone in MODEL_PATHS:
    for classifier_name, factory in CLASSIFIERS.items():
        experiment = f"{backbone}_{classifier_name}"
        classifier = factory()
        started = time.perf_counter()
        classifier.fit(FEATURES[(backbone, "train")], LABELS["train"])
        training_seconds = time.perf_counter() - started
        y_val = LABELS["val"]
        pred = classifier.predict(FEATURES[(backbone, "val")])
        scores, score_type = _accident_scores(classifier, FEATURES[(backbone, "val")])
        metrics = _metrics(y_val, pred, scores)
        fitted[experiment] = {
            "backbone": backbone, "classifier_name": classifier_name,
            "classifier": classifier, "training_seconds": training_seconds,
        }
        validation_rows.append({
            "experiment": experiment, "backbone": backbone, "classifier": classifier_name,
            "validation_accuracy": metrics["accuracy"],
            "validation_precision_accident": metrics["precision"],
            "validation_recall_accident": metrics["recall"],
            "validation_f1_accident": metrics["f1"],
            "validation_roc_auc_accident": metrics["roc_auc"],
            "training_seconds": training_seconds, "positive_class": "Accident (label 0)",
        })
        print(f"{experiment}: validation accuracy={metrics['accuracy']:.4f}, "
              f"Accident F1={metrics['f1']:.4f}, Accident ROC-AUC={metrics['roc_auc']}")

validation_metrics = pd.DataFrame(validation_rows)
selected = validation_metrics.sort_values(
    ["validation_f1_accident", "validation_accuracy", "validation_roc_auc_accident", "experiment"],
    ascending=[False, False, False, True], na_position="last",
).iloc[0]
BEST_EXPERIMENT = selected["experiment"]
print("Validation-selected configuration:", BEST_EXPERIMENT)

result_rows = []
for experiment, item in fitted.items():
    classifier, backbone = item["classifier"], item["backbone"]
    y_test, x_test = LABELS["test"], FEATURES[(backbone, "test")]
    started = time.perf_counter()
    predictions = classifier.predict(x_test)
    prediction_seconds = time.perf_counter() - started
    scores, score_type = _accident_scores(classifier, x_test)
    metrics = _metrics(y_test, predictions, scores)
    val = validation_metrics.loc[validation_metrics["experiment"] == experiment].iloc[0]
    row = {
        "experiment": experiment, "backbone": backbone, "classifier": item["classifier_name"],
        "validation_accuracy": val["validation_accuracy"],
        "validation_precision_accident": val["validation_precision_accident"],
        "validation_recall_accident": val["validation_recall_accident"],
        "validation_f1_accident": val["validation_f1_accident"],
        "validation_roc_auc_accident": val["validation_roc_auc_accident"],
        "test_accuracy": metrics["accuracy"], "test_precision_accident": metrics["precision"],
        "test_recall_accident": metrics["recall"], "test_f1_accident": metrics["f1"],
        "test_roc_auc_accident": metrics["roc_auc"],
        "training_seconds": item["training_seconds"], "test_prediction_seconds": prediction_seconds,
        "test_images": len(y_test), "positive_class": "Accident (label 0)",
        "selected_by_validation": experiment == BEST_EXPERIMENT,
    }
    result_rows.append(row)
    safe = experiment.lower()
    joblib.dump(classifier, CLASSIFIERS_DIR / f"{safe}.joblib", compress=3)
    report = classification_report(y_test, predictions, labels=[0, 1], target_names=CLASS_NAMES,
                                   zero_division=0, output_dict=True)
    with (REPORTS_DIR / f"{safe}_classification_report.json").open("w", encoding="utf-8") as file:
        json.dump({"experiment": experiment, "positive_class": "Accident (label 0)",
                   "class_mapping": CLASS_TO_LABEL, "report": report}, file, indent=2)
    matrix = confusion_matrix(y_test, predictions, labels=[0, 1])
    figure, axis = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(matrix, display_labels=CLASS_NAMES).plot(
        ax=axis, values_format="d", colorbar=False)
    axis.set_title(f"{experiment} — held-out test")
    figure.tight_layout()
    figure.savefig(CONFUSION_DIR / f"{safe}_confusion_matrix.png", dpi=150)
    plt.close(figure)
    pd.DataFrame({
        "file": [path.relative_to(PROJECT_ROOT).as_posix() for path in FILES_BY_SPLIT["test"]],
        "true_label": [LABEL_TO_CLASS[int(label)] for label in y_test],
        "true_label_id": y_test,
        "predicted_label": [LABEL_TO_CLASS[int(label)] for label in predictions],
        "predicted_label_id": predictions,
        "accident_score": scores if scores is not None else np.nan,
        "score_type": score_type or "unavailable",
    }).to_csv(PREDICTIONS_DIR / f"{safe}_predictions.csv", index=False)
    print(f"{experiment}: test accuracy={metrics['accuracy']:.4f}, "
          f"Accident F1={metrics['f1']:.4f}, prediction time={prediction_seconds:.4f}s")

classical_metrics = pd.DataFrame(result_rows)
classical_metrics.to_csv(RESULTS_DIR / "classical_ml_metrics.csv", index=False)
print("Saved all six classifiers, reports, predictions, confusion matrices, and metrics.")

## Comparison outputs

Classical metrics and existing direct-CNN metrics are recorded separately. The combined comparison includes `evaluation_protocol` and `positive_class`: archived CNN notebooks calculate binary metrics for class 1 (`Non Accident`), while classical experiments explicitly report Accident as positive (class 0). The repository-level `results/metrics.csv` is left untouched.

In [ ]:
comparison_path = RESULTS_DIR / "classical_ml_comparison.csv"
classical_metrics.sort_values(
    ["validation_f1_accident", "test_f1_accident"], ascending=[False, False]
).to_csv(comparison_path, index=False)

cnn_files = {
    "MobileNetV2": PROJECT_ROOT / "results" / "mobilenetv2_metrics.csv",
    "ResNet50": PROJECT_ROOT / "results" / "resnet50_metrics.csv",
}
cnn_rows = []
for backbone, path in cnn_files.items():
    if not path.is_file():
        raise FileNotFoundError(f"Existing CNN metrics file not found: {path}")
    frame = pd.read_csv(path)
    required = {"accuracy", "precision", "recall", "f1_score", "roc_auc", "test_images"}
    if len(frame) != 1 or not required.issubset(frame.columns):
        raise ValueError(f"Unexpected metrics schema in {path}.")
    row = frame.iloc[0]
    cnn_rows.append({
        "model_family": "Saved CNN direct classifier", "backbone": backbone,
        "classifier": "CNN sigmoid head", "experiment": f"{backbone} direct CNN",
        "accuracy": row["accuracy"], "precision": row["precision"], "recall": row["recall"],
        "f1_score": row["f1_score"], "roc_auc": row["roc_auc"],
        "training_seconds": row.get("training_seconds", np.nan),
        "test_prediction_seconds": row.get("test_prediction_seconds", np.nan),
        "test_images": row["test_images"], "positive_class": "Non Accident (label 1; CNN default)",
        "evaluation_protocol": "Archived direct CNN sigmoid predictions and saved metrics CSV",
    })
classical_rows = [{
    "model_family": "Classical classifier on saved CNN features",
    "backbone": row["backbone"], "classifier": row["classifier"],
    "experiment": row["experiment"], "accuracy": row["test_accuracy"],
    "precision": row["test_precision_accident"], "recall": row["test_recall_accident"],
    "f1_score": row["test_f1_accident"], "roc_auc": row["test_roc_auc_accident"],
    "training_seconds": row["training_seconds"],
    "test_prediction_seconds": row["test_prediction_seconds"], "test_images": row["test_images"],
    "positive_class": row["positive_class"],
    "evaluation_protocol": "Classifier on frozen features from saved accident-trained CNN; held-out test split",
} for _, row in classical_metrics.iterrows()]
combined = pd.DataFrame(cnn_rows + classical_rows)
combined_path = RESULTS_DIR / "cnn_and_classical_ml_comparison.csv"
combined.to_csv(combined_path, index=False)

figure, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
plot_data = classical_metrics.sort_values("test_f1_accident", ascending=False)
for axis, metric, title in zip(
    axes,
    ["test_accuracy", "test_f1_accident", "test_roc_auc_accident"],
    ["Test accuracy", "Test F1 (Accident positive)", "Test ROC-AUC (Accident positive)"],
):
    axis.barh(plot_data["experiment"], plot_data[metric], color="#3973ac")
    axis.set(title=title, xlim=(0, 1))
    axis.grid(axis="x", alpha=0.25)
    axis.invert_yaxis()
figure.tight_layout()
figure.savefig(RESULTS_DIR / "classical_ml_comparison.png", dpi=160)
plt.close(figure)
display(classical_metrics[[
    "experiment", "validation_f1_accident", "test_accuracy", "test_precision_accident",
    "test_recall_accident", "test_f1_accident", "test_roc_auc_accident",
    "training_seconds", "test_prediction_seconds",
]])
display(combined[["model_family", "experiment", "accuracy", "f1_score", "roc_auc",
                  "positive_class", "evaluation_protocol"]])
print("Saved:", comparison_path, combined_path, RESULTS_DIR / "classical_ml_comparison.png")

## Final experiment summary

This table shows the validation ranking used to select the best configuration and the held-out test results for all six experiments. Test metrics are shown for comparison only; they were not used for selection or tuning.

In [ ]:
columns = [
    "experiment", "validation_accuracy", "validation_f1_accident",
    "validation_roc_auc_accident", "test_accuracy", "test_precision_accident",
    "test_recall_accident", "test_f1_accident", "test_roc_auc_accident",
    "training_seconds", "test_prediction_seconds", "selected_by_validation",
]
summary = classical_metrics.sort_values(
    ["validation_f1_accident", "validation_accuracy", "validation_roc_auc_accident", "experiment"],
    ascending=[False, False, False, True], na_position="last",
)
display(summary[columns])
best = summary.loc[summary["experiment"] == BEST_EXPERIMENT].iloc[0]
print(
    f"Best validation-selected experiment: {BEST_EXPERIMENT}\n"
    f"Validation Accident F1: {best['validation_f1_accident']:.4f}; "
    f"validation accuracy: {best['validation_accuracy']:.4f}; "
    f"held-out test Accident F1: {best['test_f1_accident']:.4f}"
)
print("All classical ML artifacts:", RESULTS_DIR)